In [1]:
from pathlib import Path
from PIL import Image, ImageOps
import xml.etree.ElementTree as ET
from collections import Counter
import csv
import math

# ==================================================
# 1. CẤU HÌNH
# ==================================================

IMAGE_DIR = Path("data/train/Japan/images")
XML_DIR = Path("data/train/Japan/annotations/xmls")
OUTPUT_DIR = Path("data/train/Japan/processed_raw")

CLEAN_DIR = OUTPUT_DIR / "clean"
AMBIGUOUS_DIR = OUTPUT_DIR / "ambiguous"
MANIFEST_PATH = OUTPUT_DIR / "manifest.csv"

VALID_CLASSES = {"D00", "D10", "D20", "D40"}

CONTEXT_RATIO = 0.15
OUTPUT_SIZE = 224

# Nếu một box khác lớp nằm trong crop và phần giao chiếm
# từ 30% diện tích box đó trở lên thì xem là ambiguous
AMBIGUOUS_OVERLAP_THRESHOLD = 0.30

for group_dir in [CLEAN_DIR, AMBIGUOUS_DIR]:
    for class_name in sorted(VALID_CLASSES):
        (group_dir / class_name).mkdir(
            parents=True,
            exist_ok=True
        )


# ==================================================
# 2. HÀM HỖ TRỢ
# ==================================================

def clip_box(box, image_width, image_height):
    xmin, ymin, xmax, ymax = box

    xmin = max(0, min(float(xmin), image_width))
    ymin = max(0, min(float(ymin), image_height))
    xmax = max(0, min(float(xmax), image_width))
    ymax = max(0, min(float(ymax), image_height))

    return xmin, ymin, xmax, ymax


def get_square_crop_bounds(
    box,
    image_width,
    image_height,
    context_ratio=0.15
):
    """
    Tạo khung vuông quanh bounding box:
    - Giữ trọn damage.
    - Thêm bối cảnh.
    - Ưu tiên sử dụng vùng ảnh thật xung quanh.
    """

    xmin, ymin, xmax, ymax = box

    box_width = xmax - xmin
    box_height = ymax - ymin

    center_x = (xmin + xmax) / 2
    center_y = (ymin + ymax) / 2

    # Cạnh vuông đủ chứa box và thêm bối cảnh
    side = math.ceil(
        max(box_width, box_height)
        * (1 + 2 * context_ratio)
    )

    # Nếu ảnh đủ lớn để tạo khung vuông
    if side <= image_width and side <= image_height:
        left = round(center_x - side / 2)
        top = round(center_y - side / 2)

        # Dịch khung vào trong ảnh nếu vượt biên
        left = max(0, min(left, image_width - side))
        top = max(0, min(top, image_height - side))

        right = left + side
        bottom = top + side

        return (
            int(left),
            int(top),
            int(right),
            int(bottom)
        )

    # Nếu box quá lớn, lấy vùng mở rộng tối đa trong ảnh
    pad_x = box_width * context_ratio
    pad_y = box_height * context_ratio

    left = max(0, math.floor(xmin - pad_x))
    top = max(0, math.floor(ymin - pad_y))
    right = min(image_width, math.ceil(xmax + pad_x))
    bottom = min(image_height, math.ceil(ymax + pad_y))

    return int(left), int(top), int(right), int(bottom)


def pad_to_square(image, fill=(114, 114, 114)):
    """
    Padding ảnh thành hình vuông nếu không thể lấy khung
    vuông hoàn toàn từ ảnh gốc.
    """

    width, height = image.size
    side = max(width, height)

    pad_left = (side - width) // 2
    pad_right = side - width - pad_left

    pad_top = (side - height) // 2
    pad_bottom = side - height - pad_top

    return ImageOps.expand(
        image,
        border=(
            pad_left,
            pad_top,
            pad_right,
            pad_bottom
        ),
        fill=fill
    )


def intersection_ratio(crop_box, other_box):
    """
    Tính phần diện tích của other_box nằm trong crop.
    """

    crop_left, crop_top, crop_right, crop_bottom = crop_box
    xmin, ymin, xmax, ymax = other_box

    intersection_width = max(
        0,
        min(crop_right, xmax) - max(crop_left, xmin)
    )

    intersection_height = max(
        0,
        min(crop_bottom, ymax) - max(crop_top, ymin)
    )

    intersection_area = (
        intersection_width * intersection_height
    )

    other_area = max(1, (xmax - xmin) * (ymax - ymin))

    return intersection_area / other_area


def contains_other_class(
    crop_box,
    objects,
    target_index,
    target_class
):
    """
    Kiểm tra crop có chứa damage khác lớp rõ rệt không.
    """

    crop_left, crop_top, crop_right, crop_bottom = crop_box

    for other_index, other_object in enumerate(objects):
        if other_index == target_index:
            continue

        other_class = other_object["class_name"]

        if other_class == target_class:
            continue

        other_box = other_object["box"]
        xmin, ymin, xmax, ymax = other_box

        other_center_x = (xmin + xmax) / 2
        other_center_y = (ymin + ymax) / 2

        center_inside = (
            crop_left <= other_center_x <= crop_right
            and crop_top <= other_center_y <= crop_bottom
        )

        overlap = intersection_ratio(
            crop_box,
            other_box
        )

        if (
            center_inside
            or overlap >= AMBIGUOUS_OVERLAP_THRESHOLD
        ):
            return True

    return False


# ==================================================
# 3. XỬ LÝ TOÀN BỘ XML
# ==================================================

xml_files = sorted(XML_DIR.glob("*.xml"))

counts = Counter()
missing_images = 0
invalid_xml = 0
invalid_boxes = 0
ignored_labels = Counter()

manifest_rows = []

print(f"Tổng số XML: {len(xml_files)}")

for xml_number, xml_path in enumerate(xml_files, start=1):

    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        invalid_xml += 1
        continue

    filename = root.findtext("filename")

    if not filename:
        filename = f"{xml_path.stem}.jpg"

    # Chỉ lấy tên file, phòng trường hợp XML chứa đường dẫn
    filename = Path(filename).name
    image_path = IMAGE_DIR / filename

    if not image_path.exists():
        missing_images += 1
        continue

    with Image.open(image_path) as opened_image:
        image = opened_image.convert("RGB")

    image_width, image_height = image.size

    # Đọc tất cả object trước
    objects = []

    for obj in root.findall("object"):
        class_name = obj.findtext("name", "").strip()
        box_node = obj.find("bndbox")

        if box_node is None:
            invalid_boxes += 1
            continue

        try:
            box = (
                float(box_node.findtext("xmin")),
                float(box_node.findtext("ymin")),
                float(box_node.findtext("xmax")),
                float(box_node.findtext("ymax"))
            )
        except (TypeError, ValueError):
            invalid_boxes += 1
            continue

        box = clip_box(
            box,
            image_width,
            image_height
        )

        xmin, ymin, xmax, ymax = box

        if xmax <= xmin or ymax <= ymin:
            invalid_boxes += 1
            continue

        objects.append({
            "class_name": class_name,
            "box": box
        })

    # Tạo crop cho từng object thuộc bốn lớp chính
    for object_index, target_object in enumerate(objects):

        class_name = target_object["class_name"]

        if class_name not in VALID_CLASSES:
            ignored_labels[class_name] += 1
            continue

        target_box = target_object["box"]

        crop_box = get_square_crop_bounds(
            target_box,
            image_width,
            image_height,
            CONTEXT_RATIO
        )

        ambiguous = contains_other_class(
            crop_box=crop_box,
            objects=objects,
            target_index=object_index,
            target_class=class_name
        )

        left, top, right, bottom = crop_box

        crop = image.crop(
            (left, top, right, bottom)
        )

        # Nếu không vuông vì box quá lớn hoặc sát biên
        crop = pad_to_square(crop)

        # Resize từ vuông sang vuông nên không méo tỷ lệ
        crop = crop.resize(
            (OUTPUT_SIZE, OUTPUT_SIZE),
            Image.Resampling.LANCZOS
        )

        group_name = (
            "ambiguous"
            if ambiguous
            else "clean"
        )

        save_root = (
            AMBIGUOUS_DIR
            if ambiguous
            else CLEAN_DIR
        )

        output_name = (
            f"{xml_path.stem}"
            f"_obj{object_index}"
            f"_{class_name}.jpg"
        )

        output_path = (
            save_root
            / class_name
            / output_name
        )

        crop.save(
            output_path,
            quality=95,
            subsampling=0
        )

        counts[f"{group_name}_{class_name}"] += 1

        xmin, ymin, xmax, ymax = target_box

        manifest_rows.append({
            "source_image": filename,
            "source_xml": xml_path.name,
            "object_index": object_index,
            "class_name": class_name,
            "xmin": round(xmin),
            "ymin": round(ymin),
            "xmax": round(xmax),
            "ymax": round(ymax),
            "crop_left": left,
            "crop_top": top,
            "crop_right": right,
            "crop_bottom": bottom,
            "ambiguous": ambiguous,
            "output_path": output_path.as_posix()
        })

    if xml_number % 1000 == 0:
        print(
            f"Đã xử lý {xml_number}/{len(xml_files)} XML"
        )


# ==================================================
# 4. LƯU MANIFEST
# ==================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "source_image",
    "source_xml",
    "object_index",
    "class_name",
    "xmin",
    "ymin",
    "xmax",
    "ymax",
    "crop_left",
    "crop_top",
    "crop_right",
    "crop_bottom",
    "ambiguous",
    "output_path"
]

with open(
    MANIFEST_PATH,
    "w",
    newline="",
    encoding="utf-8-sig"
) as csv_file:

    writer = csv.DictWriter(
        csv_file,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(manifest_rows)


# ==================================================
# 5. KẾT QUẢ
# ==================================================

print("\nHOÀN THÀNH")

for class_name in sorted(VALID_CLASSES):
    print(
        f"{class_name}: "
        f"clean={counts[f'clean_{class_name}']}, "
        f"ambiguous={counts[f'ambiguous_{class_name}']}"
    )

print(f"\nXML lỗi: {invalid_xml}")
print(f"Bounding box lỗi: {invalid_boxes}")
print(f"Ảnh không tìm thấy: {missing_images}")
print(f"Nhãn bị bỏ qua: {dict(ignored_labels)}")
print(f"Manifest: {MANIFEST_PATH}")

# Kiểm tra kích thước đầu ra
output_sizes = set()

for crop_path in OUTPUT_DIR.rglob("*.jpg"):
    with Image.open(crop_path) as crop_image:
        output_sizes.add(crop_image.size)

print(f"Kích thước đầu ra: {output_sizes}")

Tổng số XML: 10506
Đã xử lý 1000/10506 XML
Đã xử lý 2000/10506 XML
Đã xử lý 3000/10506 XML
Đã xử lý 4000/10506 XML
Đã xử lý 5000/10506 XML
Đã xử lý 6000/10506 XML
Đã xử lý 7000/10506 XML
Đã xử lý 8000/10506 XML
Đã xử lý 9000/10506 XML
Đã xử lý 10000/10506 XML

HOÀN THÀNH
D00: clean=3150, ambiguous=899
D10: clean=2468, ambiguous=1511
D20: clean=3452, ambiguous=2746
D40: clean=1976, ambiguous=267

XML lỗi: 0
Bounding box lỗi: 1
Ảnh không tìm thấy: 0
Nhãn bị bỏ qua: {'D44': 3995, 'D43': 736, 'D50': 3553}
Manifest: data\train\Japan\processed_raw\manifest.csv
Kích thước đầu ra: {(224, 224)}


In [3]:
from pathlib import Path

CLEAN_DIR = Path("data/train/Japan/processed_raw/clean")

# Danh sách ảnh lỗi cần loại bỏ
bad_files = [
    "Japan_000073_obj1_D40.jpg",
    "Japan_000655_obj0_D40.jpg",
    "Japan_006823_obj0_D40.jpg",
    "Japan_008092_obj0_D40.jpg",
    "Japan_003187_obj0_D40.jpg",
    "Japan_010664_obj0_D20.jpg",
    "Japan_005541_obj2_D40.jpg",
    "Japan_009190_obj0_D00.jpg",
    "Japan_006089_obj0_D20.jpg",
    "Japan_011217_obj1_D20.jpg",
]

removed = 0
not_found = []

for filename in bad_files:
    found = False

    # tìm trong các folder D00/D10/D20/D40
    for class_dir in CLEAN_DIR.iterdir():
        file_path = class_dir / filename

        if file_path.exists():
            file_path.unlink()
            removed += 1
            found = True
            print("Đã xóa:", file_path)
            break

    if not found:
        not_found.append(filename)


print("\nHoàn thành")
print("Số ảnh đã xóa:", removed)

if not_found:
    print("Không tìm thấy:")
    for f in not_found:
        print(f)

Đã xóa: data\train\Japan\processed_raw\clean\D40\Japan_000073_obj1_D40.jpg
Đã xóa: data\train\Japan\processed_raw\clean\D40\Japan_000655_obj0_D40.jpg
Đã xóa: data\train\Japan\processed_raw\clean\D40\Japan_006823_obj0_D40.jpg
Đã xóa: data\train\Japan\processed_raw\clean\D40\Japan_008092_obj0_D40.jpg
Đã xóa: data\train\Japan\processed_raw\clean\D40\Japan_003187_obj0_D40.jpg
Đã xóa: data\train\Japan\processed_raw\clean\D20\Japan_010664_obj0_D20.jpg
Đã xóa: data\train\Japan\processed_raw\clean\D40\Japan_005541_obj2_D40.jpg
Đã xóa: data\train\Japan\processed_raw\clean\D00\Japan_009190_obj0_D00.jpg
Đã xóa: data\train\Japan\processed_raw\clean\D20\Japan_006089_obj0_D20.jpg
Đã xóa: data\train\Japan\processed_raw\clean\D20\Japan_011217_obj1_D20.jpg

Hoàn thành
Số ảnh đã xóa: 10
